# DOJ Attorney Hiring: Postings vs. Actual Hires

This notebook looks at two questions:
1. Do DOJ attorney job postings on USAJobs historically track actual hiring?
2. What has happened to both since January 20, 2025?

**Hypothesis:** DOJ may still be posting attorney jobs, but actual hiring has cratered.

## Data sources

**OPM EHRI accessions** (`impactproject/opm-ehri-data` on HuggingFace)  
Monthly counts of new federal hires by agency, occupational series, and appointment type. This is the authoritative record of who actually got hired. It's anonymous (aggregate counts only), covers the full executive branch, and runs from 2015 to present with a 1–3 month lag.

**USAJobs historical API** (`data.usajobs.gov/api/historicjoa`)  
Closed job announcements — what was posted, when, and for how long. No authentication required. Includes position title, grade, location, and (sometimes) applicant counts.

## How the connection works

Both sources use OPM's four-digit occupational series codes. Series `0905` is General Attorney — the primary attorney series at DOJ. Filtering both sources to `agency = DOJ` and `series = 0905` lets us compare postings to hires on the same population.

There will be a lag between postings and hires (federal hiring typically takes 3–6 months). We estimate that lag from pre-2025 data before looking at what happened after inauguration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import time
from datetime import datetime
from huggingface_hub import HfApi, hf_hub_download
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

INAUGURATION = pd.Timestamp('2025-01-20')

## Part 1: EHRI Accessions — Who Actually Got Hired

The EHRI dataset is structured as monthly parquet files, one per month, named `accessions/accessions_YYYYMM_v1.parquet`. We use the HuggingFace API to discover which files exist, then download and filter each to DOJ attorneys (series 0905).

In [ ]:
def get_accession_files(start_year=2018):
    """List available monthly accession parquet files from HuggingFace."""
    api = HfApi()
    all_files = api.list_repo_files(
        repo_id="impactproject/opm-ehri-data",
        repo_type="dataset"
    )
    accession_files = [
        f for f in all_files
        if f.startswith("accessions/accessions_")
        and f.endswith(".parquet")
        and int(f.split("_")[1][:4]) >= start_year
    ]
    return sorted(accession_files)

files = get_accession_files(start_year=2018)
print(f"Found {len(files)} monthly accession files from 2018 onward")
print(f"Range: {files[0]} → {files[-1]}")

In [ ]:
def load_doj_attorneys(files):
    """
    Download each monthly accession file and filter to DOJ 0905 attorneys.
    Returns a DataFrame with one row per month.
    """
    rows = []
    for fname in files:
        try:
            path = hf_hub_download(
                repo_id="impactproject/opm-ehri-data",
                filename=fname,
                repo_type="dataset"
            )
            df = pd.read_parquet(path)
            mask = (
                (df['agency'].str.upper() == 'DEPARTMENT OF JUSTICE') &
                (df['occupational_series_code'] == '0905')
            )
            subset = df[mask].copy()
            if len(subset) == 0:
                continue
            subset['count'] = pd.to_numeric(subset['count'], errors='coerce').fillna(0).astype(int)
            total = subset['count'].sum()
            yyyymm = fname.split('_')[1]
            date = pd.to_datetime(yyyymm, format='%Y%m')
            rows.append({'date': date, 'accessions': total})
        except Exception as e:
            print(f"  Skipped {fname}: {e}")
    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

print("Downloading EHRI accession files (this takes a few minutes)...")
ehri_df = load_doj_attorneys(files)
print(f"\nLoaded {len(ehri_df)} months of DOJ attorney accession data")
print(f"Date range: {ehri_df['date'].min().strftime('%Y-%m')} to {ehri_df['date'].max().strftime('%Y-%m')}")
print(f"Total DOJ attorney accessions in period: {ehri_df['accessions'].sum():,}")
ehri_df.tail(6)

## Part 2: USAJobs Postings — What Was Advertised

In [ ]:
def fetch_usajobs_doj_attorneys(start_date='2018-01-01', end_date=None, page_delay=0.4):
    """Pull all DOJ series-0905 postings from the USAJobs historical API."""
    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    BASE_URL = "https://data.usajobs.gov/api/historicjoa"
    all_items = []
    page = 1
    while True:
        params = {
            'DatePostedStart': start_date,
            'DatePostedEnd': end_date,
            'JobCategoryCode': '0905',
            'Organization': 'DEPARTMENT OF JUSTICE',
            'ResultsPerPage': 500,
            'Page': page,
        }
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        result = data.get('SearchResult', {})
        items = result.get('SearchResultItems', [])
        total = result.get('SearchResultCountAll', 0)
        for item in items:
            d = item.get('MatchedObjectDescriptor', {})
            grades = d.get('JobGrade', [])
            all_items.append({
                'control_number': d.get('PositionID'),
                'title': d.get('PositionTitle'),
                'agency': d.get('OrganizationName'),
                'location': d.get('PositionLocationDisplay'),
                'grade': grades[0].get('Code') if grades else None,
                'open_date': d.get('PositionStartDate', '')[:10],
                'close_date': d.get('PositionEndDate', '')[:10],
            })
        print(f"  Page {page}: {len(items)} items (total so far: {len(all_items)} / {total})")
        if len(all_items) >= total or not items:
            break
        page += 1
        time.sleep(page_delay)
    return all_items

print("Pulling DOJ attorney postings from USAJobs historical API...")
raw_postings = fetch_usajobs_doj_attorneys(start_date='2018-01-01')
print(f"\nTotal postings retrieved: {len(raw_postings):,}")

In [ ]:
postings_df = pd.DataFrame(raw_postings)
postings_df['open_date'] = pd.to_datetime(postings_df['open_date'], errors='coerce')
postings_df['open_month'] = postings_df['open_date'].dt.to_period('M').dt.to_timestamp()
monthly_postings = (
    postings_df.groupby('open_month')
    .size()
    .reset_index(name='postings')
    .rename(columns={'open_month': 'date'})
)
print(f"Date range: {monthly_postings['date'].min().strftime('%Y-%m')} to {monthly_postings['date'].max().strftime('%Y-%m')}")
print(f"Total postings: {monthly_postings['postings'].sum():,}")
monthly_postings.tail(8)

## Part 3: Estimating the Posting-to-Hire Lag

Before looking at the post-inauguration period, we want to know: in normal times, how long does it take between a DOJ attorney posting and an actual hire? We estimate this by computing the cross-correlation between postings and accessions on **pre-2025 data only**, then finding the lag (in months) that maximizes it.

This matters for interpreting recent data: if postings lead hires by 4 months, then a drop in postings in February 2025 wouldn't show up in accessions until June 2025.

In [ ]:
# Merge the two series
combined = pd.merge(ehri_df, monthly_postings, on='date', how='outer').sort_values('date').reset_index(drop=True)
combined = combined[combined['date'] >= '2019-01-01'].copy()
combined = combined.fillna(0)

# Use only pre-inauguration data to estimate the normal lag
pre = combined[combined['date'] < '2025-01-01'].copy()

# Cross-correlation: how much does lagging postings by k months improve its
# correlation with accessions?
MAX_LAG = 12
lags = range(0, MAX_LAG + 1)
correlations = []
for lag in lags:
    shifted = pre['postings'].shift(lag)  # shift postings forward in time
    valid = ~shifted.isna()
    r = np.corrcoef(shifted[valid], pre['accessions'][valid])[0, 1]
    correlations.append(r)

best_lag = int(np.argmax(correlations))
best_r = correlations[best_lag]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(lags), correlations, color='#4393c3', alpha=0.85)
ax.axvline(best_lag, color='#d6604d', linewidth=2, linestyle='--')
ax.set_xlabel('Lag (months postings lead accessions)')
ax.set_ylabel('Pearson r')
ax.set_title(f'Cross-correlation: USAJobs postings vs. EHRI accessions (pre-2025)\n'
             f'Best lag = {best_lag} months (r = {best_r:.2f})', fontsize=12, fontweight='bold')
ax.set_xticks(list(lags))
plt.tight_layout()
plt.savefig('lag_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest lag: {best_lag} months")
print(f"Interpretation: historically, a month's postings shows up in accessions ~{best_lag} month(s) later")

## Part 4: The Full Picture

Two views:
1. The long-run time series (2019–present), showing both raw series and postings shifted by the best lag
2. A zoom on 2024–present

In [ ]:
# Add 3-month smoothed and lag-adjusted postings
combined['accessions_3m'] = combined['accessions'].rolling(3, center=True).mean()
combined['postings_3m'] = combined['postings'].rolling(3, center=True).mean()
combined['postings_lagged'] = combined['postings'].shift(best_lag)
combined['postings_lagged_3m'] = combined['postings_lagged'].rolling(3, center=True).mean()

plot_df = combined.copy()

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

# ── Top: raw postings vs. accessions ──
ax = axes[0]
ax.plot(plot_df['date'], plot_df['postings_3m'],
        color='#2166ac', linewidth=2, label='USAJobs postings (3-mo avg)')
ax.plot(plot_df['date'], plot_df['accessions_3m'],
        color='#d6604d', linewidth=2, label='Actual hires — EHRI (3-mo avg)')
ax.fill_between(plot_df['date'], plot_df['postings'], alpha=0.08, color='#2166ac')
ax.fill_between(plot_df['date'], plot_df['accessions'], alpha=0.08, color='#d6604d')
ax.axvline(INAUGURATION, color='#444', linewidth=1.2, linestyle='--')
ax.text(INAUGURATION + pd.Timedelta(days=15), plot_df['postings_3m'].max() * 0.93,
        'Jan 20, 2025', color='#444', fontsize=9.5)
ax.set_title('DOJ Attorney Postings vs. Actual Hires (Series 0905)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Count per month')
ax.legend(frameon=False, fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Bottom: lag-adjusted postings vs. accessions ──
ax2 = axes[1]
ax2.plot(plot_df['date'], plot_df['postings_lagged_3m'],
         color='#2166ac', linewidth=2,
         label=f'Postings shifted {best_lag}mo forward (3-mo avg)')
ax2.plot(plot_df['date'], plot_df['accessions_3m'],
         color='#d6604d', linewidth=2, label='Actual hires — EHRI (3-mo avg)')
ax2.axvline(INAUGURATION, color='#444', linewidth=1.2, linestyle='--')
ax2.text(INAUGURATION + pd.Timedelta(days=15), plot_df['postings_lagged_3m'].dropna().max() * 0.93,
         'Jan 20, 2025', color='#444', fontsize=9.5)
ax2.set_title(f'Same data, postings shifted by estimated lag ({best_lag} months)',
              fontsize=12)
ax2.set_ylabel('Count per month')
ax2.legend(frameon=False, fontsize=10)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('doj_attorney_hiring.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: doj_attorney_hiring.png")

In [ ]:
# Zoom: 2024 onward — bar chart, side by side
recent = plot_df[plot_df['date'] >= '2024-01-01'].copy()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(recent['date'] - pd.Timedelta(days=8), recent['postings'],
       width=12, color='#2166ac', alpha=0.85, label='USAJobs postings')
ax.bar(recent['date'] + pd.Timedelta(days=8), recent['accessions'],
       width=12, color='#d6604d', alpha=0.85, label='Actual hires (EHRI)')
ax.axvline(INAUGURATION, color='#333', linewidth=1.5, linestyle='--', zorder=5)
ax.text(INAUGURATION + pd.Timedelta(days=5), ax.get_ylim()[1] * 0.95,
        'Jan 20, 2025', color='#333', fontsize=9.5, va='top')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
ax.set_ylabel('Count per month')
ax.set_title('DOJ Attorney Postings vs. Hires: 2024–Present\n'
             f'Note: hires typically lag postings by ~{best_lag} months',
             fontsize=12, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('doj_attorney_hiring_recent.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: doj_attorney_hiring_recent.png")

## Notes and Caveats

**The lag**  
The cross-correlation above estimates the typical posting-to-hire delay using pre-inauguration data. This is just an empirical estimate — it reflects DOJ's historical process time and can vary by position type and time of year. When interpreting recent accessions data, remember that the most recent months reflect hiring decisions made several months earlier.

**EHRI data lag**  
OPM publishes EHRI data with a 1–3 month delay, so the most recent accessions months may be incomplete.

**What's not captured**  
- Some DOJ attorney positions are filled outside USAJobs (SES, Schedule A, lateral transfers, honors program conversions)
- EHRI counts all new hires into 0905 at DOJ regardless of whether they came through a competitive posting
- The 0905 series at DOJ spans entry-level to senior litigators across all divisions

**Data sources**  
- EHRI: [`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data) on HuggingFace, updated daily  
- USAJobs: [`data.usajobs.gov/api/historicjoa`](https://developer.usajobs.gov/API-Reference/GET-api-historicjoa), no auth required  
- Methodology: [`abigailhaddad/federal-data-guide`](https://github.com/abigailhaddad/federal-data-guide)